In [1]:
from empi_pyspark_utils.spark_support.context import create_local_context
from empi_environments.env_qa import *
import pyspark.sql.functions as f
import warnings
import plotly.express as px

warnings.filterwarnings("ignore")

spark = create_local_context(cassandra_host = CASSANDRA_HOST,cpus = 2,driver_memory = '512M')

:: loading settings :: url = jar:file:/home/daniel.las/work/workspace-crypto/oda/notebooks/venv/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/daniel.las/.ivy2/cache
The jars for the packages stored in: /home/daniel.las/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
com.github.jnr#jnr-posix added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-17098f28-6265-4609-bf9f-113457f8f232;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.2.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.2.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in local-m2-cache
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhist

# Load data

In [ ]:
from functions import read_ts,read_jira_issues,normalize_jira_issues,\
    read_gitlab_projects,read_gitlab_mrs,calculate_mrs_duration,count_mrs_comments,read_gitlab_commits

# TS
ts = read_ts(spark)
ts_jira_hours = ts.where(f.col('jira_id').isNotNull())
ts_other_hours = ts.where(f.col('jira_id').isNull())
users_jira_hours = ts_jira_hours.groupBy('month','user').agg(f.sum('hours').alias('jira_hours'))
users_other_hours = ts_other_hours.groupBy('month','user').agg(f.sum('hours').alias('other_hours'))
users_hours = users_jira_hours\
.join(users_other_hours,['month','user'],'left')\
.withColumn('total_hours',f.col('jira_hours') + f.col('other_hours'))\
.withColumn('other_hours_pct',f.col('other_hours')/f.col('total_hours')*100)\
.orderBy(f.col('month').desc(),f.col('user'))

# JIRA
issues = read_jira_issues(spark)
issues_normalized = normalize_jira_issues(issues)
issues_open = issues_normalized.where(f.col('resolutiondate').isNull()).orderBy(f.col('created').desc())
issues_completed = issues_normalized.where(f.col('resolutiondate').isNotNull()).orderBy(f.col('created').desc())
issues_completed_by_month = issues_completed.groupBy('resolutiondate_month','type').count()
issues_hours = ts.groupBy('jira_id').agg(f.sum('hours').alias('issue_total_hours')).withColumnRenamed('jira_id','key')

# Gitlab projects
projects = read_gitlab_projects(spark)

# Gitlab merge requests
mrs = read_gitlab_mrs(spark)
mrs_count = mrs.groupBy('month').agg(f.count('*').alias('count')).orderBy(f.col('month').desc())
project_mrs = projects.join(mrs,projects.id == mrs.project_id)
mrs_count_by_type = project_mrs.groupBy('month','project_type').agg(f.count('*').alias('count')).orderBy(f.col('month').desc())
mrs_duration = calculate_mrs_duration(project_mrs)
mrs_comments = count_mrs_comments(project_mrs)

# Gitlab commits
commits = read_gitlab_commits(spark)
project_commits = commits.join(projects,commits.projectId == projects.id)
commits_by_project_type = calculate_commits_stats(project_commits)

# Completed issues monthly

In [ ]:
px.bar(issues_completed_by_month.toPandas(),x='resolutiondate_month',y='count',color='type',title='Completed issues')

# Heavy open issues

In [ ]:
from functions import filter_heavy_open_issues

work_hours_threshold = 16
skip_status = ['Backlog','Delivery','Delivered']

filter_heavy_open_issues(issues_open, ts, issues_hours, work_hours_threshold, skip_status).toPandas()

# Merged merge requests monthly

In [ ]:
px.bar(mrs_count_by_type.toPandas(),x='month',y='count',color='project_type',title='MR count by project type')

# Merge requests duration monthly

In [ ]:
px.bar(mrs_duration.toPandas(),x='month',y='duration_days_pct50',color='project_type',title='MR duration pct50',log_y=True,barmode='group')

In [ ]:
px.bar(mrs_duration.toPandas(),x='month',y='duration_days_pct75',color='project_type',title='MR duration pct75',log_y=True,barmode='group')

# Merge requests comments monthly

In [ ]:
px.bar(mrs_comments.toPandas(),x='month',y='comments',color='project_type',title='MR comments')

# Commits

In [ ]:
px.bar(commits_by_project_type.toPandas(),x='month',y='additions',color='project_type',title='Code additions',log_y=True)

In [ ]:
px.bar(commits_by_project_type.toPandas(),x='month',y='deletions',color='project_type',title='Deletions',log_y=True)

In [ ]:
from datetime import datetime,timedelta
import pandas as pd

pd.set_option('display.max_rows', None)

def past_date(days):
    return (datetime.now() - timedelta(days = 180)).replace(hour = 0,minute = 0, second = 0, microsecond = 0)

def user_issues(user,past_days = 180):
    return issues_normalized\
        .join(ts,ts.jira_id == issues_normalized.key, 'left')\
        .join(issues_hours,issues_hours.key == issues_open.key)\
        .where(f.col('user')==user)\
        .where(f.col('status_changed_at') > past_date(past_days))\
        .groupBy(
            issues_normalized.key,
            f.col('summary'),
            f.col('status'),
            f.col('type'),
            f.col('priority'),
            f.col('size'),
            f.col('size_points'),
            f.col('status_changed_at'),
            f.col('issue_total_hours'))\
        .agg(f.sum('hours').alias('user_total_hours'))\
        .orderBy(f.col('status_changed_at').desc())

def user_other_hours(user):
    return ts_other_hours\
        .where(f.col('user')==user)\
        .groupBy('month','task_desc')\
        .agg(f.sum('hours').alias('other_hours'))\
        .orderBy(f.col('month').desc())

def user_issues_hours_monthly(user):
    return ts\
        .where(f.col('user') == user)\
        .groupBy('month','jira_id','task_desc')\
        .agg(f.sum('hours').alias('hours'))\
        .join(issues_normalized,ts.jira_id == issues_normalized.key)\
        .select('month','jira_id','summary','status','hours')\
        .orderBy(f.col('month').desc(),f.col('jira_id'))

def user_mrs_duration(user):
    return project_mrs\
        .where(f.col('author.username') == user)\
        .where(f.col('project_type') == 'code')\
        .groupBy('month')\
        .agg(
            f.avg('duration_days').alias('duration_days_avg'),
            f.min('duration_days').alias('duration_days_min'),
            f.max('duration_days').alias('duration_days_max'),
            f.expr('percentile(duration_days, array(0.25,0.5,0.75,0.9,0.99))').alias('duration_days_pct'))\
        .withColumn('duration_days_pct25',f.col('duration_days_pct')[0])\
        .withColumn('duration_days_pct50',f.col('duration_days_pct')[1])\
        .withColumn('duration_days_pct75',f.col('duration_days_pct')[2])\
        .withColumn('duration_days_pct90',f.col('duration_days_pct')[3])\
        .withColumn('duration_days_pct99',f.col('duration_days_pct')[4])

def user_code_commits(user):
    return project_commits\
        .where(f.col('committer') == user)\
        .where(f.col('project_type') == 'code')\
        .groupBy('month')\
        .agg(
            f.count(commits.id).alias('count'),
            f.sum('additions').alias('additions'),
            f.sum('deletions').alias('deletions'))\
        .orderBy(f.col('month').desc())

# KBa

In [ ]:
user_issues('KBa').toPandas()

# PAn

In [ ]:
user_issues('PAn').toPandas()

In [ ]:
px.scatter(user_mrs_duration('piotr.antas').toPandas(),x='month',y=['duration_days_pct25','duration_days_pct50','duration_days_pct75'], log_y=True,title='Code MR duration')

In [ ]:
px.bar(user_code_commits('piotr.antas').toPandas(),x='month',y=['additions','deletions'],log_y=True,title='Commits',barmode='group')

In [ ]:
users_hours.where(f.col('user')=='PAn').toPandas()

In [ ]:
user_other_hours('PAn').toPandas()

# TGa

In [ ]:
user_issues('TGa').toPandas()

In [ ]:
user_issues_hours_monthly('TGa').toPandas()

In [ ]:
px.bar(user_issues_hours_monthly('TGa').toPandas(),x='month',y='hours',color='jira_id',hover_data={'summary':True})

In [ ]:
px.scatter(user_mrs_duration('tomasz.gac').toPandas(),x='month',y=['duration_days_pct25','duration_days_pct50','duration_days_pct75'], log_y=True,title='Code MR duration')

In [ ]:
px.bar(user_code_commits('tomasz.gac').toPandas(),x='month',y=['additions','deletions'],log_y=True,title='Commits',barmode='group')

In [ ]:
users_hours.where(f.col('user')=='TGa').toPandas()

In [ ]:
user_other_hours('TGa').toPandas()

# MRy

In [ ]:
user_issues('MRy').toPandas()

In [ ]:
user_issues_hours_monthly('MRy').toPandas()

In [ ]:
px.bar(user_issues_hours_monthly('MRy').toPandas(),x='month',y='hours',color='jira_id',hover_data={'summary':True})

In [ ]:
px.scatter(user_mrs_duration('mateusz.ryl').toPandas(),x='month',y=['duration_days_pct25','duration_days_pct50','duration_days_pct75'], log_y=True,title='Code MR duration')

In [ ]:
px.bar(user_code_commits('mateusz.ryl').toPandas(),x='month',y=['additions','deletions'],log_y=True,title='Commits',barmode='group')

In [ ]:
users_hours.where(f.col('user')=='MRy').toPandas()

In [ ]:
user_other_hours('MRy').toPandas()